# Part 2: NeRF 3D Reconstruction

This notebook orchestrates Part 2 training and rendering using helper modules:
- `rendering.py`
- `nerf_model.py`
- `train_part2.py`
- `part2_utils.py`

Start with a smoke run, then scale to full training.

In [ ]:
from pathlib import Path
import numpy as np
import torch

In [3]:
from part2_utils import ensure_dir, plot_training_curves, save_depth_png, save_rgb_png, set_seed
from train_part2 import build_part2_data, render_test_trajectory, train_nerf_part2

In [9]:
# Core configuration (mid run profile)
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_path": "lego_200x200.npz",
    "output_dir": "images/output/part2",
    "hidden_dim": 256,
    "n_layers": 8,
    "pos_freqs": 10,
    "dir_freqs": 4,
    "n_coarse": 32,
    "n_fine": 32,
    "batch_rays": 1024,
    "n_steps": 20,
    "eval_every": 30,
    "chunk_size": 4096,
    "lr": 5e-4,
    "near_override": 2.0,
    "far_override": 6.0,
}

set_seed(CFG["seed"])
ensure_dir(CFG["output_dir"])
CFG

{'seed': 42,
 'device': 'cpu',
 'data_path': 'lego_200x200.npz',
 'output_dir': 'images/output/part2',
 'hidden_dim': 256,
 'n_layers': 8,
 'pos_freqs': 10,
 'dir_freqs': 4,
 'n_coarse': 32,
 'n_fine': 32,
 'batch_rays': 1024,
 'n_steps': 20,
 'eval_every': 30,
 'chunk_size': 4096,
 'lr': 0.0005,
 'near_override': 2.0,
 'far_override': 6.0}

In [10]:
# Quick data sanity check (no training)
data = build_part2_data(CFG["data_path"], device=CFG["device"])
print("K shape:", tuple(data["k"].shape))
print("Train rays:", len(data["train_dataset"]))
print("Val images:", tuple(data["val_images"].shape))
print("Test poses:", tuple(data["test_c2ws"].shape))

K shape: (3, 3)
Train rays: 4000000
Val images: (10, 200, 200, 3)
Test poses: (60, 4, 4)


In [17]:
# Launch Viser server for camera/ray/sample visualization
import sys
import subprocess
from pathlib import Path


def make_viser_command(
    cfg: dict,
    num_cameras: int = 1,
    camera_start_idx: int = 0,
    num_rays: int = 300,
    num_samples_along_ray: int = 64,
    port: int = 8080,
) -> list[str]:
    return [
        sys.executable,
        "visualize_viser.py",
        "--data_path", cfg["data_path"],
        "--near", str(cfg["near_override"]),
        "--far", str(cfg["far_override"]),
        "--num_samples_along_ray", str(num_samples_along_ray),
        "--num_rays", str(num_rays),
        "--num_cameras", str(num_cameras),
        "--camera_start_idx", str(camera_start_idx),
        "--device", cfg["device"],
        "--port", str(port),
    ]


def start_viser_server(
    cfg: dict,
    num_cameras: int = 1,
    camera_start_idx: int = 0,
    num_rays: int = 300,
    num_samples_along_ray: int = 64,
    port: int = 8080,
):
    cmd = make_viser_command(
        cfg,
        num_cameras=num_cameras,
        camera_start_idx=camera_start_idx,
        num_rays=num_rays,
        num_samples_along_ray=num_samples_along_ray,
        port=port,
    )
    proc = subprocess.Popen(cmd, cwd=str(Path.cwd()))
    print(f"Viser started with PID={proc.pid}")
    print(f"Open http://localhost:{port}")
    print(f"Showing {num_cameras} camera(s), start index {camera_start_idx}")
    return proc


# One-line launch; tweak args as needed.
viser_proc = start_viser_server(
    CFG,
    num_cameras=60,
    camera_start_idx=0,
    num_rays=300,
    num_samples_along_ray=64,
    port=8080,
)

Viser started with PID=39480
Open http://localhost:8080
Showing 60 camera(s), start index 0


In [15]:
# Stop Viser server launched from this notebook
try:
    viser_proc.terminate()
    viser_proc.wait(timeout=5)
    print("Viser stopped.")
except NameError:
    print("No running viser_proc found in this notebook session.")
except Exception as e:
    print("Could not stop cleanly:", e)

Viser stopped.


In [11]:
# Smoke training run (increase n_steps and batch_rays later)
results = train_nerf_part2(
    data_path=CFG["data_path"],
    output_dir=CFG["output_dir"],
    device=CFG["device"],
    seed=CFG["seed"],
    hidden_dim=CFG["hidden_dim"],
    n_layers=CFG["n_layers"],
    pos_freqs=CFG["pos_freqs"],
    dir_freqs=CFG["dir_freqs"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    n_steps=CFG["n_steps"],
    batch_rays=CFG["batch_rays"],
    lr=CFG["lr"],
    eval_every=CFG["eval_every"],
    chunk_size=CFG["chunk_size"],
    near_override=CFG["near_override"],
    far_override=CFG["far_override"],
)
results["metrics"]

[train_nerf_part2] start device=cpu steps=20 batch_rays=1024 coarse=32 fine=32 near=2.000 far=6.000 eval_every=30
[train_nerf_part2] step 1/20 loss=0.198607 coarse=0.079104 fine=0.190696 iter=3.856s rays_per_sec=266
[train_nerf_part2] step 2/20 loss=0.194627 coarse=0.082429 fine=0.186384 iter=3.596s rays_per_sec=285
[train_nerf_part2] step 3/20 loss=0.189450 coarse=0.078884 fine=0.181561 iter=3.698s rays_per_sec=277
[train_nerf_part2] step 4/20 loss=0.185795 coarse=0.090129 fine=0.176783 iter=3.690s rays_per_sec=277
[train_nerf_part2] step 5/20 loss=0.179257 coarse=0.093979 fine=0.169859 iter=3.546s rays_per_sec=289
[train_nerf_part2] step 6/20 loss=0.177449 coarse=0.099374 fine=0.167511 iter=3.458s rays_per_sec=296
[train_nerf_part2] step 7/20 loss=0.173703 coarse=0.082778 fine=0.165425 iter=3.558s rays_per_sec=288
[train_nerf_part2] step 8/20 loss=0.163733 coarse=0.085296 fine=0.155204 iter=3.390s rays_per_sec=302
[train_nerf_part2] step 9/20 loss=0.167159 coarse=0.194219 fine=0.1477

{'best_psnr': 10.635948181152344,
 'best_step': 20,
 'final_loss': 0.0995035320520401,
 'near': 2.0,
 'far': 6.0,
 'n_steps': 20,
 'batch_rays': 1024,
 'n_coarse': 32,
 'n_fine': 32,
 'val_psnr_hist': [10.635948181152344],
 'eval_steps': [20],
 'total_seconds': 146.2055124000035,
 'avg_seconds_per_step': 7.310275620000175,
 'log_every': 1}

In [12]:
# Save training curves
curve_dir = Path(CFG["output_dir"]) / "curves"
plot_training_curves(
    loss_hist=results["loss_hist"],
    eval_steps=results["eval_steps"],
    val_psnr_hist=results["val_psnr_hist"],
    output_dir=curve_dir,
)
print("Saved curves to", curve_dir)

Saved curves to images\output\part2\curves


In [13]:
# Render 60 test views to NPY (RGB + depth)
models = results["models"]
data = results["data"]
image_hw = (data["val_images"].shape[1], data["val_images"].shape[2])

render_test_trajectory(
    model_coarse=models["coarse"],
    model_fine=models["fine"],
    k=data["k"],
    test_c2ws=data["test_c2ws"],
    image_hw=image_hw,
    output_dir=CFG["output_dir"],
    near=results["metrics"]["near"],
    far=results["metrics"]["far"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    chunk_size=CFG["chunk_size"],
    device=CFG["device"],
)
print("Saved test trajectory npy files.")

KeyboardInterrupt: 

In [13]:
# Optional: convert NPY outputs to PNG files
rgb_npy_dir = Path(CFG["output_dir"]) / "test_rgb_npy"
depth_npy_dir = Path(CFG["output_dir"]) / "test_depth_npy"
rgb_png_dir = Path(CFG["output_dir"]) / "test_renders"
depth_png_dir = Path(CFG["output_dir"]) / "depth_maps"

rgb_png_dir.mkdir(parents=True, exist_ok=True)
depth_png_dir.mkdir(parents=True, exist_ok=True)

for npy_path in sorted(rgb_npy_dir.glob("view_*.npy")):
    img = np.load(npy_path)
    save_rgb_png(img, rgb_png_dir / f"{npy_path.stem}.png")

for npy_path in sorted(depth_npy_dir.glob("view_*.npy")):
    dep = np.load(npy_path)
    save_depth_png(dep, depth_png_dir / f"{npy_path.stem}.png")

print("Saved PNG outputs:", rgb_png_dir, depth_png_dir)

Saved PNG outputs: images\output\part2\test_renders images\output\part2\depth_maps
